# Model 1 — Autoregressive Date Generator
Run on a **T4 GPU** Colab: Runtime → Change runtime type → T4 GPU

In [ ]:
# ── 1. Clone & install ────────────────────────────────────────────────────────
REPO = "https://github.com/SalmaSherif7070/Conditional-Date-Generation-Using-Deep-Generative-Models"
!git clone {REPO} repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# ── 2. GPU check ─────────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NOT available — set Runtime → T4 GPU')

In [ ]:
# ── 3. Create __init__.py files ───────────────────────────────────────────────
import os
for pkg in ['src', 'src/model_1', 'src/model_2', 'src/model_3', 'src/model_4']:
    os.makedirs(pkg, exist_ok=True)
    p = os.path.join(pkg, '__init__.py')
    if not os.path.exists(p):
        open(p, 'w').close()
print('✓ __init__.py ready')

In [ ]:
# ── 4. Write minimal src/model_1/visualization.py ────────────────────────────
viz_code = '''
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

STYLE  = {"figure.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False}
C_TR   = "#4C72B0"
C_VAL  = "#DD8452"
COLORS = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]

def _save(fig, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

def plot_loss_curves(history, save_dir):
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(history["epochs"], history["train_loss"], lw=2, color=C_TR,  label="Train")
        ax.plot(history["epochs"], history["val_loss"],   lw=2, color=C_VAL, label="Val", ls="--")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.set_title("Model 1 – Loss (Train vs Val)"); ax.legend(); ax.grid(alpha=0.3)
        _save(fig, os.path.join(save_dir, "model1_loss_curves.png"))

def plot_loss_log_scale(history, save_dir):
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.semilogy(history["epochs"], history["train_loss"], lw=2, color=C_TR,  label="Train")
        ax.semilogy(history["epochs"], history["val_loss"],   lw=2, color=C_VAL, label="Val", ls="--")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (log)")
        ax.set_title("Model 1 – Loss Log Scale"); ax.legend(); ax.grid(alpha=0.3, which="both")
        _save(fig, os.path.join(save_dir, "model1_loss_log.png"))

def plot_condition_breakdown(metrics, save_dir):
    labels = ["Day of Week", "Month", "Leap Year", "Decade", "All (CSR)"]
    values = [metrics["dow_acc"], metrics["mon_acc"], metrics["leap_acc"],
              metrics["decade_acc"], metrics["csr"]]
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(labels, values, color=COLORS, edgecolor="white")
        ax.set_ylabel("Accuracy"); ax.set_title("Model 1 – Per-Condition Accuracy")
        ax.set_ylim(0, 1.15)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.02,
                    f"{val:.1%}", ha="center", va="bottom", fontsize=10)
        ax.grid(axis="y", alpha=0.3)
        _save(fig, os.path.join(save_dir, "model1_condition_breakdown.png"))
'''
with open('src/model_1/visualization.py', 'w') as f:
    f.write(viz_code.strip())
print('✓ src/model_1/visualization.py written')

In [ ]:
# ── 5. Patch config.py — 100 epochs, flat output paths ───────────────────────
config_src = '''
from dataclasses import dataclass

@dataclass
class ModelConfig:
    cond_dim: int = 128
    max_decade: int = 300

@dataclass
class TrainConfig:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 2e-3
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model2Config:
    cond_dim: int = 128
    max_decade: int = 300
    z_dim: int = 64

@dataclass
class Train2Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr_g: float = 1e-4
    lr_d: float = 4e-4
    n_critic: int = 2
    lambda_gp: float = 10.0
    tau_start: float = 2.0
    tau_end: float = 0.5
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model3Config:
    cond_dim: int = 128
    max_decade: int = 300
    z_dim: int = 64

@dataclass
class Train3Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 1e-3
    beta_max: float = 0.5
    beta_warmup_frac: float = 0.5
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model4Config:
    cond_dim: int = 128
    max_decade: int = 300
    hidden_dim: int = 512
    n_layers: int = 4

@dataclass
class Train4Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 1e-4
    n_mcmc_steps: int = 60
    mcmc_step_size: float = 0.1
    mcmc_noise: float = 0.005
    replay_buffer_size: int = 10_000
    replay_prob: float = 0.95
    l2_reg: float = 1.0
    grad_clip: float = 1.0
    val_split: float = 0.2
    seed: int = 42

@dataclass
class PathConfig:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path2Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    generator_path: str = "output/generator.pt"
    discriminator_path: str = "output/discriminator.pt"
    encoder_path: str = "output/encoder.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path3Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path4Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"
'''
with open('src/config.py', 'w') as f:
    f.write(config_src.strip())
print('✓ src/config.py patched (100 epochs, flat output)')

In [ ]:
# ── 6. Patch main.py to use model_1 visualization ────────────────────────────
with open('main.py', 'r') as f:
    src = f.read()

# Replace Model 1 visualization imports to use model_1.visualization
src = src.replace(
    'from src.visualization import plot_loss_curves, plot_condition_breakdown, plot_loss_log_scale',
    'from src.model_1.visualization import plot_loss_curves, plot_loss_log_scale, plot_condition_breakdown'
)

with open('main.py', 'w') as f:
    f.write(src)
print('✓ main.py patched')

In [ ]:
# ── 7. Create output directories ─────────────────────────────────────────────
import os
for d in ['output/figures']:
    os.makedirs(d, exist_ok=True)
print('✓ Directories ready')

In [ ]:
# ── 8. Train ──────────────────────────────────────────────────────────────────
!python main.py train

In [ ]:
# ── 9. Evaluate ───────────────────────────────────────────────────────────────
!python main.py evaluate

In [ ]:
# ── 10. Predict ───────────────────────────────────────────────────────────────
!python main.py predict \
    -i data/raw/example_input.txt \
    -o output/predictions.txt

print('\nFirst 10 predictions:')
with open('output/predictions.txt') as f:
    for i, line in enumerate(f):
        if i >= 10: break
        print(line, end='')

In [ ]:
# ── 11. Display figures ───────────────────────────────────────────────────────
from IPython.display import Image, display
import glob
figs = sorted(glob.glob('output/figures/*.png'))
print(f'Found {len(figs)} figures:')
for path in figs:
    print('\n', path)
    display(Image(path))

In [ ]:
# ── 12. Zip & download ────────────────────────────────────────────────────────
import zipfile, os
ZIP = 'output/model1_outputs.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir('output/figures')):
        zf.write(f'output/figures/{fn}', f'figures/{fn}')
    if os.path.exists('output/predictions.txt'):
        zf.write('output/predictions.txt', 'predictions.txt')
    if os.path.exists('output/weights.pt'):
        zf.write('output/weights.pt', 'weights.pt')
print(f'✓ ZIP ({os.path.getsize(ZIP)/1e6:.1f} MB) → {ZIP}')
from google.colab import files
files.download(ZIP)